# Issue #232 — causal SP-backed divergence support (R2 fresh)

Contract: BJ-UNIFIED-DIVERGENCE-SUPPORT-001 / v1, mode: candidate.
R2 rework after EV REJECT (PR #239) — method-level fresh rebuild.

## R2 defect closures
- **C1** causal gate: every decision flows through `causal.causal_available` (`observed_at <= decision_epoch <= kickoff`); R1 published 365 post-kickoff matches as causal.
- **C2** READY gate: pure truth table in `disposition.py`; R1 never required the independent-source interface nor every chronological role.
- **C3** official 3SP: fail-closed LABEL_ONLY (never expanded into triplets).
- **E1** AUDITED_SHA: external reviewed HEAD required; zero placeholder rejected.
- **E2** semantic verifier: independent recomputation of causality/devig/disposition/evidence/binding/schema/files; 9/9 tamper mutations blocked.
- **E3** replay: #232-native manifest/replay (no #229 resolver).
- **E4** waterfall: LINE_RECONCILIATION for every source + issue/date/line dimensions; no hard-coded TRUE.

## 1. Build the immutable package

Rebuilds `results/bj232_divergence_support_v2` from the committed upstream packages and verifies it with the independent semantic verifier.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from divergence_support import builder, contract, verifier

AUDITED_SHA = 'a3cb398c20bef944065a8a60fc33d36e2d8cd38e'
receipt = builder.build(ROOT, audited_sha=AUDITED_SHA)
print('disposition:', receipt.disposition)
print('support:', json.dumps(receipt.support_summary, indent=2))

disposition: UNRECOVERABLE_WITH_CURRENT_SOURCES
support: {
  "qbj_sp_complete_matches": 0,
  "qbj_eu_sp_complete_matches": 0,
  "qbj_power_sp_complete_matches": 0,
  "cohort_issue_support": {
    "fit": 0,
    "tune": 0,
    "calibration": 0
  }
}


## 2. Support waterfall (REQ-S06)

Per-source stage ledger: RAW_ROWS → IDENTITY_JOIN → LINE_RECONCILIATION → TIMESTAMP_GATE → COMPLETE_TRIPLET → CAUSAL_INTERSECTION.

In [2]:
waterfall = receipt.waterfall
print('waterfall rows:', len(waterfall))
print(waterfall.to_string(index=False))

waterfall rows: 34
        source                         source_key               stage  count   kind unit                       reason
 Q_BJ_SP_DEVIG bj214_selected_market_observations            RAW_ROWS  16496   PASS  row                             
 Q_BJ_SP_DEVIG bj214_selected_market_observations       IDENTITY_JOIN   5332   DROP  row            NO_IDENTITY_MATCH
 Q_BJ_SP_DEVIG bj214_selected_market_observations       IDENTITY_JOIN  11164   PASS  row                             
 Q_BJ_SP_DEVIG bj214_selected_market_observations LINE_RECONCILIATION  11164   PASS  row                             
 Q_BJ_SP_DEVIG bj214_selected_market_observations      TIMESTAMP_GATE   9333   DROP  row     POST_KICKOFF_OBSERVATION
 Q_BJ_SP_DEVIG bj214_selected_market_observations      TIMESTAMP_GATE     65   DROP  row         DUPLICATE_SOURCE_ROW
 Q_BJ_SP_DEVIG bj214_selected_market_observations      TIMESTAMP_GATE   1766   PASS  row                             
 Q_BJ_SP_DEVIG bj214_selected_market_

## 3. Recovered decision-epoch SP triplets (C1 verified)

Every row passes the full frozen gate; the causal counts must match the EV independent recomputation (376 matches / 16 issues / 1,128 cells).

In [3]:
triplets = receipt.recovery_sp_triplets
print('triplet rows:', len(triplets), '| issues:', triplets['issue'].nunique() if len(triplets) else 0,
      '| cells:', len(triplets) * 3)
print('causal all true:', bool(triplets['causal'].all()) if len(triplets) else 'n/a')
print('epoch<=kickoff all true:', bool(triplets['gate_epoch_le_kickoff'].all()) if len(triplets) else 'n/a')
print('years:', sorted(triplets['year'].unique()) if len(triplets) else [])
print()
print(receipt.recovery_receipts.to_string(index=False))

triplet rows: 376 | issues: 16 | cells: 1128
causal all true: True
epoch<=kickoff all true: True
years: [np.int64(2026)]

    interface  recovered_triplet_cells  recovered_issue_count    reason
Q_BJ_SP_DEVIG                     1128                     16 RECOVERED


## 4. Issue-level modelability (REQ-S08)

Interface summary by chronological cohort (fit ≤2021, tune 2022, calibration 2023).

In [4]:
summary = receipt.modelability.get('interface_summary', pd.DataFrame())
print(summary.to_string(index=False))
print()
n_support = receipt.modelability.get('n_support_qbj_sp', pd.DataFrame())
print('N-support (Q_BJ x SP):')
print(n_support.to_string(index=False))

        interface  fit_issues  fit_matches  tune_issues  tune_matches  calibration_issues  calibration_matches  2024_plus_issues  2024_plus_matches
        Q_BJ_x_SP           0            0            0             0                   0                    0                 0                  0
   Q_BJ_x_EU_x_SP           0            0            0             0                   0                    0                 0                  0
Q_BJ_x_POWER_x_SP           0            0            0             0                   0                    0                 0                  0

N-support (Q_BJ x SP):
 n  issues_with_ge_n  fraction_of_issues
 1                 0                 0.0


## 5. Independent-source reconciliation (contract item 5)

In [5]:
print('POWER reconciled cells:', len(receipt.reconcile_power), '| issues:',
      receipt.reconcile_power['issue'].nunique() if len(receipt.reconcile_power) else 0)
print('EU reconciled cells:', len(receipt.reconcile_eu), '| issues:',
      receipt.reconcile_eu['issue'].nunique() if len(receipt.reconcile_eu) else 0)
print()
print(receipt.reconcile_receipts.to_string(index=False))

POWER reconciled cells: 0 | issues: 0
EU reconciled cells: 0 | issues: 0

          interface  reconciled_cells  reconciled_issues  line_reconciled  translation_available
Q_BJ_x_POWER_BJLINE                 0                  0           4674.0                    NaN
   Q_BJ_x_EU_BJLINE                 0                  0              NaN               192795.0


## 6. Zero-overlap diagnosis (contract item 2)

In [6]:
print(receipt.diagnosis.to_string(index=False))

interface              root_class                                                                   evidence  anchor_issue_count  anchor_cell_count  sp_raw_rows  sp_identity_drops  sp_timestamp_drops  sp_triplet_drops  eu_raw_rows  eu_line_drops  eu_timestamp_drops  power_raw_rows  power_line_drops  power_timestamp_drops  official_3sp_raw_drops official_3sp_reason
Q_BJ_x_SP ISSUE_CALENDAR_MISMATCH SP rows fail the identity join against the #203 issue calendar; drops=5332                  33                228        16496               5332                9398               236       332718         139923              190953            1558                 0                   1480                       1    RAW_ASSET_ABSENT


## 7. REQ-S evidence (no hard-coded TRUE)

In [7]:
print(receipt.evidence[['requirement', 'pass']].to_string(index=False))
print()
print('debt closure:', json.dumps({k: v for k, v in receipt.debt_closure.items() if k != 'problems'}, indent=2))

requirement  pass
    REQ-S01  True
    REQ-S02  True
    REQ-S03  True
    REQ-S04  True
    REQ-S05  True
    REQ-S06  True
    REQ-S07  True
    REQ-S08  True
    REQ-S09  True
    REQ-S10  True

debt closure: {
  "E1_exact_reviewed_sha_binding": true,
  "E2_physical_schema_domains": true,
  "E3_crash_restart_journal": true,
  "E4_append_only_protocol_log": true,
  "all_closed": true
}


## 8. Independent verification of the published package

In [8]:
pkg = contract.package_path(ROOT)
hashes = verifier.verify_package_hashes(pkg)
problems = verifier.verify_semantics(pkg, expected_sha=AUDITED_SHA)
print(f'{len(hashes)} files hash-verified')
if problems:
    for p in problems:
        print('SEMANTIC PROBLEM:', p)
else:
    print('SEMANTIC CHECKS PASS')

28 files hash-verified
SEMANTIC CHECKS PASS
